In [ ]:
def get_coords_heavy(mol):
    labels = [l for l, c in zip(mol.labels, mol.coord) if l != 'H']
    coords = [c for l, c in zip(mol.labels, mol.coord) if l != 'H']
    return labels, coords

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

scopepath = '/Users/sergivela/Documents/SCOPE/Program/'
sys.path.append(scopepath)

import Scope
from Scope import *

from ase import Atoms

In [ ]:
sys_path = "/Users/sergivela/Documents/SCOPE/Database_SCO/6-Systems_V3/"
figures_path = "/Users/sergivela/Documents/SCOPE/Database_SCO/Figures_Paper/"

In [ ]:
finished_systems = load_binary(sys_path+"finished_systems.npy")

# SOAP

In [ ]:
# Step 1: Collect all atomic species (C, O, N, Se...) used across the dataset
all_species = set()

for sys in finished_systems:
    refcode = sys.refcode
    if refcode == "NAPYFE10" or refcode == "WOJLUC" or refcode == "XUPDUH02": 
        continue
    found, HS_state = find_state(sys.HS_ref_mol, "b3lyp_opt")
    mol = HS_state.get_moleclist()[0]
    labels, coords = get_coords_heavy(mol)
    structure = Atoms(labels=labels, positions=coords)
    all_species.update(structure.get_chemical_labels())

global_species = sorted(all_species)  # or use a manually defined order
soap = SOAP(species=global_species,r_cut=5.0,n_max=8,l_max=6,sigma=1.0,rbf="gto",sparse=False,periodic=False,average="outer") # per-atom SOAP vectors

In [ ]:
def compute_soap3(mol, soap, debug: int=0):
    symbols, coords = get_coords_heavy(mol)
    # 1. Get full structure (without hydrogens)
    full_structure = Atoms(symbols=symbols, positions=coords)

    # 2. Get coordination sphere: Fe + coordinating atoms
    metal = mol.metals[0]  # SCOPE Metal object
    coordination_sphere = metal.get_coord_sphere()  # Returns list of atoms (not ASE)

    # Convert coordination sphere to ASE Atoms object
    cs_symbols = [atom.label for atom in coordination_sphere]
    cs_coords  = [atom.coord for atom in coordination_sphere]
    cs_atoms   = Atoms(symbols=cs_symbols, positions=cs_coords)

    # Convert metal to ASE Atoms object
    metal_symbol = metal.label
    metal_coord  = metal.coord
    metal_atom   = Atoms(symbols=[metal_symbol], positions=[metal_coord])

    # Combine them
    atoms_fe_n = cs_atoms + metal_atom

    # 3. Get positions of Fe-N atoms
    fe_n_positions = atoms_fe_n.get_positions()

    # 4. Match positions to indices in full_structure
    fe_n_indices = []
    for i, atom in enumerate(full_structure):
        for pos in fe_n_positions:
            if np.allclose(atom.position, pos, atol=0.1):  # Adjust tolerance if needed
                fe_n_indices.append(i)
                break

    # 5. Create Atoms objects for Fe-N and non-Fe-N parts
    atoms_inner = full_structure[fe_n_indices]
    atoms_outer = full_structure[[i for i in range(len(full_structure)) if i not in fe_n_indices]]

    # 6. Define SOAP species
    soap_inner = soap.create(atoms_inner)
    soap_outer = soap.create(atoms_outer)

    # 7. Compute SOAP for the coordination sphere centered around the metal (soap_n_coord_center)
    # The coordination sphere can be thought of as a cluster of atoms surrounding the metal
    soap_n_coord_center = soap.create(cs_atoms)  # SOAP descriptor for the coordination environment

    return soap_fe_n, soap_non_fe_n, soap_n_coord_center